# Topic: SQL ROW_NUMBER() Pattern

## Definition (30-second explanation)
* `ROW_NUMBER()` is a window function that assigns a unique, sequential integer to every row within a result set or partition, starting from 1.
* Unlike other ranking functions, it never produces duplicate values; every single row gets its own unique number, regardless of ties.

## Why Interviewers Ask This
* To test your ability to handle "get exactly one row per group" scenarios, which are extremely common in real-world data pipelines.
* To verify you understand the difference between window functions and standard aggregate functions.
* To see if you understand query execution order, specifically that window functions cannot be directly filtered in a `WHERE` clause.

## Core Concepts
* **PARTITION BY:** Divides the result set into groups; `ROW_NUMBER()` resets to 1 at the start of each new partition.
* **ORDER BY (inside OVER):** Defines how rows are ordered within each partition *before* numbers are assigned.
* **CTE / Subquery Requirement:** You must wrap the window function in a Common Table Expression (CTE) or subquery to filter on its result.

## When to Use
* Deduplicating records while keeping the most recent or most important row per group.
* Selecting the top-N rows per group (e.g., top 3 salespeople per region).
* Implementing pagination in APIs or dashboards (e.g., page 1: rows 1-10).
* Finding the first or last event per user/session in event logs.

## Advantages
* Guarantees uniqueness—sets it apart from `RANK()` and `DENSE_RANK()`.
* Perfect for intelligent deduplication and returning deterministic results when tied with a robust `ORDER BY` clause.

## Limitations
* Not suitable when ties should receive the same rank number.
* Can be slow when working with extremely large datasets without proper indexing on the `PARTITION BY` and `ORDER BY` columns.

## Common Comparisons
* **vs RANK():** `RANK()` gives the same number to ties and skips the next ranks (e.g., 1, 1, 3). `ROW_NUMBER()` always gives unique numbers (1, 2, 3).
* **vs DENSE_RANK():** `DENSE_RANK()` gives the same number to ties but does *not* skip the next rank (e.g., 1, 1, 2).
* **vs Correlated Subquery:** Subqueries can achieve similar filtering but often evaluate slower on large datasets and do not elegantly handle tie-breaking for single-row returns.

## Common Interview Traps
* **Forgetting ORDER BY inside OVER():** Without it, row numbers are assigned arbitrarily, leading to non-deterministic results.
* **Using in WHERE directly:** You CANNOT use `ROW_NUMBER()` in a `WHERE` clause directly.
* **Missing PARTITION BY:** Without it, it numbers ALL rows in the entire table instead of resetting per group.
* **Wrong ORDER BY direction:** Forgetting `DESC` when you want the "latest" or "highest" value to get `rn=1`.

## SQL Syntax 
```sql
WITH RankedData AS (
  SELECT 
    column1,
    ROW_NUMBER() OVER (
      PARTITION BY column_to_group_by 
      ORDER BY column_to_sort_by DESC
    ) AS rn
  FROM table_name
)
SELECT * FROM RankedData WHERE rn = 1;
```

## 45-Second Interview Answer
"ROW_NUMBER() is my go-to window function for deduplication and top-N-per-group problems. It assigns a unique, sequential integer to rows within specific partitions, guaranteeing no duplicates even if there are ties. Because window functions evaluate after the WHERE clause, I always implement this pattern by computing the row number inside a CTE, ordering by a tie-breaker like a timestamp, and then querying that CTE to filter where the row number equals 1."